In the following code, we visually demonstrate the impact of sampling on the Discrete Fourier Transform (DFT) using a sine wave as a base example. The code allows users to additionally analyze a second sine wave, or a waveform generated through amplitude modulation (AM), frequency modulation (FM), or the sum or combination of both sine waves. Dotted lines in the plots represent the time steps or sampling intervals.

1. **Aliasing and Oversampling**: Explore how aliasing and oversampling affect the waveform in the time domain and its corresponding DFT in the frequency domain.
2. **Sampling Rate and Nyquist Frequency**: Compare the sampling rate (SR) and the Nyquist frequency with the number of available sample points within the DFT to understand their interrelationship.
3. **Frequency Resolution**: Investigate what affects the frequency resolution more - is it the sampling rate or the total length of the signal?
4. **Amplitude and Phase Information**: Observe the amplitude and phase information captured in the DFT and how it corresponds to the properties of the original signal.
5. **Spectral Leakage**: Visualize the effects of spectral leakage, especially when the signal's frequency components do not fall exactly on the frequency bins of the DFT.
6. **Windowing**: The use of window functions (like the Blackman window) can be explored to mitigate spectral leakage and improve the frequency resolution of the DFT.

This notebook serves as an interactive tool to understand some key concepts of digital signal processing, including sampling, aliasing, frequency resolution, spectral leakage, and windowing.


In [22]:
import numpy as np
import plotly.graph_objects as go
from scipy.signal import get_window
import plotly.io as pio
#pio.renderers.default='notebook'



# Define all parameters at the top
n = 0.1  # Time step
t_max = 2.0  # Maximum time
selected_waveforms = ['sine1']  # Waveforms to show as list: 'sine1', 'sine2', 'am', 'fm', 'sum', 'comb'
amp1, freq1, phase1 = 1.0, 3.25, 0*np.pi/2  # Parameters for sine wave 1: amp, freq, phase
amp2, freq2, phase2 = 1.0, 6, 0*np.pi/2  # Parameters for sine wave 2: amp, freq, phase

index_am = 1  # Set the modulation index for AM
index_fm = 10  # Set the modulation index for FM

apply_window = True  # Apply window function
window_type = 'hann'  # Type of window function to apply. Reference here: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.get_window.html

add_dotted_lines = True  # Add dotted lines overlay



# Calculate time vector
t = np.arange(0, t_max, n)

# Initialize waveform data dictionary
waveform_data = {}

# Calculate sine wave values
waveform_data['sine1'] = amp1 * np.sin(2*np.pi*freq1*t + phase1)
waveform_data['sine2'] = amp2 * np.sin(2*np.pi*freq2*t + phase2)

# AM with modulation index as ratio
waveform_data['am'] = (amp1 + index_am * waveform_data['sine2']) * np.sin(2*np.pi*freq1*t + phase1)

# FM with modulation index as ratio
waveform_data['fm'] = amp1 * np.sin(2*np.pi*(freq1 + index_fm * waveform_data['sine2'])*t + phase1)

waveform_data['sum'] = waveform_data['sine1'] + waveform_data['sine2']

# Calculate combined sine wave values
waveform_data['comb'] = np.zeros_like(t)  # Initialize combined waveform
half_period = int(t_max / (2 * n))  # Half period length in indices

for i in range(0, len(t), half_period):
    if (i // half_period) % 2 == 0:  # If even half period, use sine1
        waveform_data['comb'][i:i+half_period] = waveform_data['sine1'][i:i+half_period]
    else:  # If odd half period, use sine2
        waveform_data['comb'][i:i+half_period] = waveform_data['sine2'][i:i+half_period]

# Apply window function if enabled
if apply_window:
    window = get_window(window_type, len(t))
    for waveform in waveform_data:
        waveform_data[waveform] *= window

# Create the Plotly figure for waveforms
fig_waveforms = go.Figure()

# Plot the selected waveforms
waveform_max_min = {'max': np.amax([waveform_data[waveform] for waveform in selected_waveforms]),
                    'min': np.amin([waveform_data[waveform] for waveform in selected_waveforms])}

for waveform in selected_waveforms:
    fig_waveforms.add_trace(go.Scatter(x=t, y=waveform_data[waveform], mode='lines', name=waveform))

# Plot the window function if enabled
if apply_window:
    fig_waveforms.add_trace(go.Scatter(x=t, y=window, mode='lines', name='window - ' + window_type, line=dict(dash='dash')))

# Add vertical dotted lines to waveform plot
if add_dotted_lines:
    range_25_percent = 0.25 * (waveform_max_min['max'] - waveform_max_min['min'])
    num_lines_waveform = int(t_max / n) + 1
    for x in np.linspace(0, t_max, num_lines_waveform):
        fig_waveforms.add_shape(type="line", x0=x, y0=waveform_max_min['min'] - range_25_percent,
                                x1=x, y1=waveform_max_min['max'] + range_25_percent,
                                line=dict(color="Black", width=1, dash="dot"))

# Calculate and print sampling rate and Nyquist frequency
sampling_rate = 1 / n
nyquist_freq = sampling_rate / 2
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Nyquist frequency: {nyquist_freq} Hz")

# Print total number of available sample points
print(f"Total number of available sample points: {len(t)}")   
    
# Display the waveform plot
fig_waveforms.update_layout(title='Sine Waves', xaxis_title='Time (s)', yaxis_title='Amplitude',
                            autosize=False, width=800, height=600)
fig_waveforms.show()

# Create the DFT Magnitude and Phase Plots
fig_dft_magnitude = go.Figure()
fig_dft_phase = go.Figure()

# Find the maximum amplitude and minimum phase of the DFT across selected waveforms
max_dft_amp = 0
min_dft_phase = np.inf

for waveform in selected_waveforms:
    y = waveform_data[waveform]

    Y = np.fft.fft(y)
    freqs = np.fft.fftfreq(len(t), d=n)
    inds = np.argsort(freqs)
    freqs = freqs[inds]
    Y = Y[inds]

    max_dft_amp = max(max_dft_amp, np.max(np.abs(Y)))
    min_dft_phase = min(min_dft_phase, np.min(np.angle(Y)))

    # Add to Magnitude and Phase plots of the DFT
    fig_dft_magnitude.add_trace(go.Scatter(x=freqs, y=np.abs(Y), mode='lines', name=waveform + ' - ' + window_type if apply_window else waveform))
    fig_dft_phase.add_trace(go.Scatter(x=freqs, y=np.angle(Y), mode='lines', name=waveform + ' - ' + window_type if apply_window else waveform))

# Add vertical dotted lines to DFT plots
if add_dotted_lines:
    range_25_percent_magnitude = 0.25 * max_dft_amp
    range_25_percent_phase = 0.25 * (np.pi - min_dft_phase)
    num_lines_dft = int(t_max / n)
    for x in np.linspace(np.min(freqs), np.max(freqs), num_lines_dft):
        fig_dft_magnitude.add_shape(type="line", x0=x, y0=0, x1=x, y1=max_dft_amp + range_25_percent_magnitude, line=dict(color="Black", width=1, dash="dot"))
        fig_dft_phase.add_shape(type="line", x0=x, y0=min_dft_phase - range_25_percent_phase, x1=x, y1=np.pi + range_25_percent_phase, line=dict(color="Black", width=1, dash="dot"))

# Update layout for DFT Magnitude
fig_dft_magnitude.update_layout(title='Magnitude Spectrum of the DFT', xaxis_title='Frequency (Hz)', yaxis_title='Magnitude', autosize=False, width=800, height=600)

# Update layout for DFT Phase
fig_dft_phase.update_layout(title='Phase Spectrum of the DFT', xaxis_title='Frequency (Hz)', yaxis_title='Phase (radians)', autosize=False, width=800, height=600)

# Display the DFT Magnitude plot
fig_dft_magnitude.show()

# Display the DFT Phase plot
fig_dft_phase.show()


Sampling rate: 10.0 Hz
Nyquist frequency: 5.0 Hz
Total number of available sample points: 20
